# Tool Use / Function Calling with LangChain & LangGraph

**A standalone companion notebook — frameworks only.**

You already have a notebook that implements tool use / function calling directly on the OpenAI Python SDK, with no framework (`tool_use_function_calling.ipynb`, Part 1). This notebook assumes you've run that one and picks up exactly where it leaves off: **the same problems, solved with LangChain and LangGraph.**

Every raw-SDK concept you already know maps onto something here:

| Raw SDK concept | LangChain / LangGraph equivalent |
| --- | --- |
| `tools=[...]` JSON Schema list | `@tool` / `bind_tools` |
| `message.tool_calls[i].function.arguments` (JSON string) | `response.tool_calls[i]["args"]` (already-parsed dict) |
| Hand-written `while` loop (ReAct) | `AgentExecutor` or LangGraph `agent ⇄ tools` graph |
| Manual `tool_call_id` bookkeeping | `ToolNode` (does it for you) |
| No persistence unless you build it | LangGraph `checkpointer` |
| No pause/resume unless you build it | LangGraph `interrupt()` / `interrupt_before` |
| Manual streaming delta accumulation | `astream_events` |

Nothing here invents a new protocol — LangChain and LangGraph are convenience layers over exactly the tool-calling wire protocol you already understand. The goal of this notebook is to show you **precisely which parts are "free" once you adopt the framework, and which parts you still have to design yourself.**

Written entirely in English, runnable top to bottom, using simulated tools so most cells work offline — only cells that call a real LLM need an API key.


## 0. Setup

We install `langchain` + provider integration packages, `langgraph`, and a couple of extras used later (`langgraph-checkpoint-sqlite` for durable persistence, `langchain-anthropic` / `langchain-google-genai` for the provider-portability demo in section 1.4).

You only strictly need `OPENAI_API_KEY` to run most cells. The Anthropic/Gemini keys are only needed for section 1.4 — skip that cell if you don't have them.


In [ ]:
# %pip install --quiet langchain langchain-core langchain-openai langchain-anthropic \
#     langchain-google-genai langgraph langgraph-checkpoint-sqlite pydantic

In [ ]:
import os
import json
import time
import asyncio
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")

# Optional — only needed for the multi-provider portability demo in 1.4.
# Leave blank and skip that cell if you don't have these.
ANTHROPIC_AVAILABLE = bool(os.environ.get("ANTHROPIC_API_KEY"))
GOOGLE_AVAILABLE = bool(os.environ.get("GOOGLE_API_KEY"))

MODEL = "gpt-4.1"  # any tool-calling-capable OpenAI model works

### Optional: LangSmith tracing

Neither the Word guide nor the Part 1 notebook covers observability — worth adding here because it's the single highest-leverage production addition once you move to a framework. LangSmith is LangChain/LangGraph's tracing backend: every `.invoke()`, tool call, and graph node execution is logged automatically once these three environment variables are set — **zero code changes to your agent.**

This is optional and every other cell in this notebook works without it.


In [ ]:
# Uncomment to enable tracing (requires a free LangSmith account: https://smith.langchain.com)
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = getpass("Enter your LANGCHAIN_API_KEY: ")
# os.environ["LANGCHAIN_PROJECT"] = "tool-use-langchain-langgraph"

# With tracing on, every cell below produces a clickable trace URL in the LangSmith UI showing
# the full sequence of LLM calls, tool executions, and (in Part 2) graph node transitions —
# the framework-level equivalent of the print statements you added by hand in Part 1.

---
# PART 1 — LangChain: Defining and Binding Tools

LangChain solves one specific problem from the raw-SDK notebook: **the wire format for tool schemas and tool-call results differs by provider** (OpenAI vs Anthropic vs Gemini — see the comparison table in the Word guide, section 1.2.1). LangChain lets you define a tool once, in one Python-native shape, and reuse it across `ChatOpenAI`, `ChatAnthropic`, and `ChatGoogleGenerativeAI` unchanged.

We'll define two simulated tools — `get_weather` and `calculator` — once, and reuse them for the rest of this notebook.


In [ ]:
def get_weather(city: str, unit: str = "celsius") -> dict:
    """Simulated weather lookup — replace with a real API call in production."""
    fake_db = {"hanoi": 32, "tokyo": 24, "san francisco": 18, "new york": 21, "london": 15}
    temp_c = fake_db.get(city.lower(), 20)
    temp = temp_c if unit == "celsius" else temp_c * 9 / 5 + 32
    return {"city": city, "temperature": round(temp, 1), "unit": unit, "condition": "sunny"}


def calculator_fn(operation: str, a: float, b: float) -> dict:
    """Basic arithmetic used by several sections below (raw callable, wrapped as a tool next)."""
    ops = {"add": lambda x, y: x + y, "subtract": lambda x, y: x - y,
           "multiply": lambda x, y: x * y, "divide": lambda x, y: x / y if y != 0 else None}
    if operation not in ops:
        raise ValueError(f"Unsupported operation: {operation}")
    result = ops[operation](a, b)
    if result is None:
        raise ZeroDivisionError("Division by zero")
    return {"operation": operation, "a": a, "b": b, "result": result}

## 1.1 Defining Tools the LangChain Way

Three equivalent ways to turn a Python function into a tool-callable object. All three produce the same OpenAI-compatible JSON Schema you built by hand in Part 1 — LangChain infers it from type hints and docstrings.


In [ ]:
from langchain_core.tools import tool, StructuredTool
from pydantic import BaseModel, Field

# (a) The @tool decorator — most common. Docstring -> description, type hints -> JSON Schema.
@tool
def get_weather_tool(city: str, unit: str = "celsius") -> dict:
    """Get the current weather for a given city.
    Use this whenever the user asks about weather, temperature, or whether they need an umbrella."""
    return get_weather(city, unit)


# (b) StructuredTool.from_function — for wrapping a function you don't own / don't want to decorate.
calculator_tool = StructuredTool.from_function(
    func=calculator_fn,
    name="calculator",
    description="Perform a basic arithmetic operation: add, subtract, multiply, or divide.",
)


# (c) An explicit Pydantic args_schema — full control over validation, nested objects, and
# field-level constraints beyond what type hints alone express. Use this when you need enum-style
# constraints, the same accuracy gains as section 1.2 in the raw-SDK notebook, expressed via Pydantic.
class WeatherInput(BaseModel):
    city: str = Field(description="The city name, e.g. 'Hanoi' or 'San Francisco'")
    unit: str = Field(default="celsius", description="celsius or fahrenheit")


@tool(args_schema=WeatherInput)
def get_weather_typed(city: str, unit: str = "celsius") -> dict:
    """Get the current weather for a given city."""
    return get_weather(city, unit)


TOOLS = [get_weather_tool, calculator_tool]
print(get_weather_tool.name, "->", get_weather_tool.description)
print(json.dumps(get_weather_tool.args, indent=2))  # inferred JSON Schema — compare to Part 1's hand-written version

**Async tools** are defined identically with `async def`; LangChain dispatches to the async path automatically when you call `.ainvoke()` / `.astream()` on the agent (see section 1.12).


In [ ]:
@tool
async def get_weather_async(city: str, unit: str = "celsius") -> dict:
    """Async version of get_weather — identical signature, just `async def`."""
    await asyncio.sleep(0.1)  # simulate a real async HTTP call
    return get_weather(city, unit)

## 1.2 Binding Tools to a Model

`ChatOpenAI.bind_tools([...])` returns a new runnable with the tool schemas attached to every call — the LangChain equivalent of passing `tools=TOOLS` in the raw SDK. The first concrete convenience: `response.tool_calls` gives you a list of `{"name", "args", "id"}` dicts with **`args` already parsed** — no `json.loads()` needed.


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL)
llm_with_tools = llm.bind_tools(TOOLS)

response = llm_with_tools.invoke("What's the weather in Hanoi?")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)  # already-parsed dicts, unlike raw SDK's JSON string

## 1.3 Provider Portability

*(Not covered as runnable code in the source materials — added here because it's the entire point of the wire-format comparison table in the Word guide, section 1.2.1.)*

The same `TOOLS` list, the same `.bind_tools()` call, works unchanged against `ChatAnthropic` and `ChatGoogleGenerativeAI`. LangChain translates the tool schema into whichever provider-specific envelope (`input_schema` for Anthropic, `function_declarations` for Gemini) is required underneath, and normalizes the response back into the same `.tool_calls` shape regardless of provider.


In [ ]:
if ANTHROPIC_AVAILABLE:
    from langchain_anthropic import ChatAnthropic
    claude = ChatAnthropic(model="claude-sonnet-4-6").bind_tools(TOOLS)
    r = claude.invoke("What's the weather in Tokyo?")
    print("Anthropic tool_calls:", r.tool_calls)  # same shape as the OpenAI response above
else:
    print("Skipped — set ANTHROPIC_API_KEY to run this cell.")

if GOOGLE_AVAILABLE:
    from langchain_google_genai import ChatGoogleGenerativeAI
    gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash").bind_tools(TOOLS)
    r = gemini.invoke("What's the weather in London?")
    print("Gemini tool_calls:", r.tool_calls)  # same shape again
else:
    print("Skipped — set GOOGLE_API_KEY to run this cell.")

# The point: your tool definitions and your response-parsing code are now provider-agnostic.
# Swapping providers is a one-line change (llm = ChatAnthropic(...) instead of ChatOpenAI(...)),
# not a rewrite of your tool schemas or your tool_calls parsing logic.

## 1.4 Forcing / Restricting Tool Choice

`bind_tools(..., tool_choice=...)` is the LangChain equivalent of the raw SDK's `tool_choice` parameter (Word guide, section 1.9):


In [ ]:
# "any"/"required" — model must call some tool this turn
forced = llm.bind_tools(TOOLS, tool_choice="any")

# force one specific tool by name — useful in guided workflows where step N always uses tool X
forced_specific = llm.bind_tools(TOOLS, tool_choice="calculator")
r = forced_specific.invoke("What's the weather like today?")  # unrelated question, tool forced anyway
print(r.tool_calls)

## 1.5 `return_direct` — Skipping the Second LLM Call

*(Not in the source materials — a genuinely LangChain-specific convenience with no raw-SDK equivalent, worth knowing.)*

Normally, after a tool executes, the tool result is sent back to the model for a final natural-language turn (the last step of the loop in section 1.7 below). Setting `return_direct=True` on a tool tells `AgentExecutor` / LangGraph's prebuilt agent to **return the tool's raw output immediately**, skipping that second LLM call — useful when a tool's output is already the final answer (e.g., a lookup tool where no further reasoning adds value) and you want to save the latency/cost of a second round trip.


In [ ]:
@tool(return_direct=True)
def lookup_order_status(order_id: str) -> str:
    """Look up the shipping status of an order by ID. The result is shown to the user verbatim,
    with no further LLM paraphrasing — use return_direct for tools whose output IS the answer."""
    return f"Order {order_id} shipped on 2026-06-28, arriving 2026-07-04."

## 1.6 `with_structured_output` — Forced Single-Schema Extraction

This is the framework-native way to do what the raw SDK does with `tool_choice="required"` bound to a single-purpose tool (Word guide, section 1.9: *"useful for forcing structured extraction... where free text is never an acceptable output"*). `with_structured_output` skips the tool-call envelope entirely and returns a validated Pydantic instance directly.


In [ ]:
class TicketClassification(BaseModel):
    category: str = Field(description="One of: billing, technical, account, other")
    urgency: str = Field(description="One of: low, medium, high")

classifier = llm.with_structured_output(TicketClassification)
result = classifier.invoke("My card was charged twice and I need this fixed today.")
print(result)  # TicketClassification(category='billing', urgency='high') — a real Pydantic object, not a tool_call to parse

## 1.7 A Manual Loop with LangChain Primitives (No Agent Abstraction)

Before reaching for `AgentExecutor`, rebuild the exact ReAct loop from the raw-SDK notebook using only `bind_tools` and typed message objects — the bridge that makes the jump to frameworks non-magical.


In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

def run_manual_loop(user_input, max_iterations=8):
    messages = [HumanMessage(content=user_input)]
    tool_map = {t.name: t for t in TOOLS}
    for _ in range(max_iterations):
        ai_msg = llm_with_tools.invoke(messages)
        messages.append(ai_msg)
        if not ai_msg.tool_calls:
            return ai_msg.content, messages
        for call in ai_msg.tool_calls:
            result = tool_map[call["name"]].invoke(call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return "Max iterations reached.", messages

answer, history = run_manual_loop("What's the weather in Hanoi, and what's 12 * 7?")
print(answer)

Structurally identical to Part 1's `run_agent` — LangChain hasn't changed the protocol, only given you typed message classes and pre-parsed arguments.

## 1.8 `create_agent` — the High-Level Agent Runtime

A ready-made version of the loop above. **Note:** as of LangChain 1.0 (GA, October 2025), the old class-based `AgentExecutor` / `create_tool_calling_agent` were removed from the main `langchain` package. `create_agent` is their replacement — a single factory that builds an agent (internally, a compiled LangGraph graph) from a model, a tool list, and an optional system prompt. It accepts a `checkpointer`, `interrupt_before`, and a `middleware` list for cross-cutting concerns (human-in-the-loop, summarization, PII redaction, retries) — the successor to hand-written AgentExecutor callbacks.


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,               # a BaseChatModel instance, or a "provider:model" string
    tools=TOOLS,
    system_prompt="You are a helpful assistant.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Hanoi, and what's 12 * 7?"}]})
print(result["messages"][-1].content)

# create_agent returns a compiled LangGraph app under the hood, so everything from Part 2
# (checkpointer=, streaming, .ainvoke()) already works on it unchanged — see section 2.2,
# which covers langgraph.prebuilt.create_react_agent, the lower-level sibling of this call.


### If you specifically need the old `AgentExecutor` class

Legacy chains and agents (including `AgentExecutor`, `create_tool_calling_agent`, `create_openai_tools_agent`) still exist, but now live in a separate package: `pip install langchain-classic`, then `from langchain_classic.agents import AgentExecutor, create_tool_calling_agent`. For new code, prefer `create_agent` above — it is the maintained, actively-developed path.


### Tool-level error handling in LangChain

`handle_tool_error` turns a tool crash into an observation fed back to the model — the LangChain equivalent of the `try/except` pattern in Part 1, section 1.8.


In [ ]:
@tool(handle_tool_error=True)
def flaky_calculator(operation: str, a: float, b: float) -> dict:
    """Same as calculator, but demonstrates handle_tool_error: an exception becomes
    a message the model sees as a tool result instead of crashing the executor."""
    return calculator_fn(operation, a, b)

# calculator_fn raises ZeroDivisionError on divide-by-zero; with handle_tool_error=True,
# the executor catches it and feeds the model "Error: Division by zero" as the tool result,
# letting the model apologize or try a different approach — instead of crashing your app.

## 1.9 Retries and Timeouts

*(Not covered in the source materials — worth adding: production tools call flaky external APIs.)*

`.with_retry()` wraps any Runnable — including a tool or the LLM call itself — with exponential-backoff retry, and `.with_config(timeout=...)` bounds how long a single call may take. This is the LangChain-native equivalent of writing your own retry decorator around a tool function (which still works fine and is framework-agnostic — see the Word guide, section 2.11).


In [ ]:
resilient_weather_tool = get_weather_tool.with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=True,
)
# Use resilient_weather_tool in TOOLS/AgentExecutor exactly like the original —
# retry behavior is transparent to callers.

resilient_llm = llm.with_retry(stop_after_attempt=3)  # same pattern applies to the LLM call itself

## 1.10 Async Tools and `.ainvoke()`

LangChain dispatches to the `async def` path automatically once you invoke with `.ainvoke()` / `.astream()` — no separate code path to maintain.


In [ ]:
async def run_async_example():
    llm_async_tools = llm.bind_tools([get_weather_async, calculator_tool])
    response = await llm_async_tools.ainvoke("What's the weather in Tokyo?")
    if response.tool_calls:
        call = response.tool_calls[0]
        result = await get_weather_async.ainvoke(call["args"])
        print(result)

await run_async_example()  # in a plain .py script, wrap this in asyncio.run(...) instead

## 1.11 Part 1 Recap

| Raw SDK (Part 1 notebook) | LangChain (this notebook) |
| --- | --- |
| Hand-written JSON Schema dict | `@tool` / `StructuredTool` / `args_schema` |
| `tools=TOOLS` per-request | `llm.bind_tools([...])` — reusable runnable |
| `tool_call.function.arguments` (string, `json.loads()` yourself) | `response.tool_calls[i]["args"]` (already parsed) |
| One provider only | Same tool objects work across OpenAI / Anthropic / Gemini |
| Hand-written `try/except` per tool | `handle_tool_error=True` |
| Hand-written retry decorator | `.with_retry()` |
| `tool_choice` param, single provider format | `bind_tools(tool_choice=...)`, portable |
| N/A | `with_structured_output` for pure extraction |
| N/A | `return_direct=True` to skip the second LLM turn |

**What LangChain has *not* solved yet:** persistence across restarts, pausing for human approval, and multi-agent graphs. That's LangGraph — Part 2.


---
# PART 2 — LangGraph: The Agent Loop as an Explicit Graph

LangGraph re-expresses the ReAct `while` loop (Part 1's `run_manual_loop`, and the raw-SDK notebook's `run_agent`) as an explicit, resumable **state graph**: two node types (`agent`, `tools`) and one conditional edge. That explicitness is what unlocks everything `AgentExecutor` can't do: checkpointed persistence, human-in-the-loop interrupts, and multi-agent composition as subgraphs.

**Rule of thumb used throughout:** use LangChain's tool abstractions everywhere (low-cost, portable). Reach for LangGraph over `AgentExecutor` for anything beyond a quick prototype.


## 2.1 The Agent ⇄ Tools Graph

`MessagesState` is a prebuilt state schema: a single key, `messages`, that accumulates via a reducer (append, don't overwrite) every time a node returns new messages.


In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def call_model(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph = StateGraph(MessagesState)
graph.add_node("agent", call_model)
graph.add_node("tools", ToolNode(TOOLS))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)   # routes to "tools" or END
graph.add_edge("tools", "agent")                         # loop back after executing

app = graph.compile()
result = app.invoke({"messages": [("user", "What's the weather in Hanoi?")]})
print(result["messages"][-1].content)

Map this directly back to the raw-SDK loop: `"agent"` node = the API call; `ToolNode` = your `execute_tool` dispatch loop; `tools_condition` = the `if not msg.tool_calls: return` check; the `"tools" → "agent"` edge = looping back to the top of the `while`.

`ToolNode` automatically: executes every tool call in a turn (parallel calls included), catches tool exceptions and returns them as `ToolMessage` content, and matches `tool_call_id`s correctly — all the bookkeeping from Part 1, done for you.


In [ ]:
# Visualize the graph structure (Mermaid rendering — falls back gracefully if unsupported).
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(app.get_graph().draw_ascii())

## 2.2 The Prebuilt `create_react_agent`

For the common case, LangGraph ships the exact graph above as one call.


In [ ]:
from langgraph.prebuilt import create_react_agent

react_agent = create_react_agent(llm, tools=TOOLS)
react_agent.invoke({"messages": [("user", "What's the weather in Hanoi?")]})

# Use the hand-built graph from 2.1 when you need custom nodes (e.g. a guardrails-check node
# before "agent"); use create_react_agent when the standard shape is all you need — you can
# always "eject" into the explicit graph later since both compile to the same structure.

**Naming note:** `langgraph.prebuilt.create_react_agent` (used here) and `langchain.agents.create_agent` (section 1.8) are siblings, not the same function. Both compile to a LangGraph app with the same agent/tools shape; `create_agent` sits one layer higher and adds the middleware system (built-in human-in-the-loop, summarization, PII redaction, retries). Reach for `create_react_agent` when you're already working in LangGraph and want the plain prebuilt graph; reach for `create_agent` when you want LangChain's higher-level, middleware-driven agent construction.


## 2.3 Custom State Schema and Reducers

*(Not covered as runnable code in the source materials — important for anything beyond a pure chat loop.)*

`MessagesState` is convenient but limited to one field. Real agents usually need to track more: a user ID, a running budget, a scratchpad. Define your own `TypedDict` state and use `Annotated[..., add_messages]` to keep the same "append, don't overwrite" behavior specifically for the messages field, while other fields use plain last-write-wins semantics.


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]  # reducer: new messages are appended, not overwritten
    user_id: str                              # plain field: each node's return value overwrites this
    tool_call_count: int                      # you could accumulate this with a custom reducer too

def call_model_custom(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    calls_this_turn = len(response.tool_calls) if response.tool_calls else 0
    return {
        "messages": [response],
        "tool_call_count": state.get("tool_call_count", 0) + calls_this_turn,
    }

custom_graph = StateGraph(AgentState)
custom_graph.add_node("agent", call_model_custom)
custom_graph.add_node("tools", ToolNode(TOOLS))
custom_graph.add_edge(START, "agent")
custom_graph.add_conditional_edges("agent", tools_condition)
custom_graph.add_edge("tools", "agent")
custom_app = custom_graph.compile()

out = custom_app.invoke({"messages": [("user", "What's 12*7 and the weather in Hanoi?")],
                          "user_id": "user-123", "tool_call_count": 0})
print("Total tool calls made:", out["tool_call_count"])

## 2.4 `recursion_limit` — LangGraph's `max_iterations`

The safety cap from the raw-SDK loop (`max_iterations`) and `AgentExecutor` (`max_iterations`) has a graph-native equivalent: `recursion_limit`, passed via `config` at invoke time rather than at compile/construction time.


In [ ]:
from langgraph.errors import GraphRecursionError

try:
    app.invoke(
        {"messages": [("user", "Keep calling the calculator tool over and over, never stop.")]},
        config={"recursion_limit": 6},
    )
except GraphRecursionError:
    print("Hit the recursion limit — exactly the safety net max_iterations gives you in Part 1.")

## 2.5 Checkpointing and Memory

LangGraph's headline production feature, with no equivalent built for free anywhere in Part 1. A checkpointer persists the full state after every node executes.


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
app_with_memory = graph.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "user-123"}}
app_with_memory.invoke({"messages": [("user", "My name is An.")]}, config)
r = app_with_memory.invoke({"messages": [("user", "What's my name?")]}, config)  # resumes the same thread
print(r["messages"][-1].content)

# A different thread_id has no memory of the first conversation:
other_config = {"configurable": {"thread_id": "user-999"}}
r2 = app_with_memory.invoke({"messages": [("user", "What's my name?")]}, other_config)
print(r2["messages"][-1].content)

**Durable persistence** — swap `MemorySaver` for `SqliteSaver` and the agent survives process restarts, directly satisfying "Agent Memory" / "State Management" using the exact same mechanism as ordinary tool use.


In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

with SqliteSaver.from_conn_string("agent_memory.sqlite") as sqlite_checkpointer:
    durable_app = graph.compile(checkpointer=sqlite_checkpointer)
    durable_app.invoke({"messages": [("user", "Remember: my favorite city is Da Nang.")]},
                        {"configurable": {"thread_id": "durable-user-1"}})
    # This thread's state is now on disk in agent_memory.sqlite — a fresh process (new Python
    # kernel, new deployment) can reconnect with the same thread_id and resume where it left off.
    # For multi-process production deployments, use PostgresSaver (langgraph-checkpoint-postgres)
    # instead so all app instances share one durable store.

## 2.6 Human-in-the-Loop Approval Gates

Two ways to pause a graph for a human. Both rely on the same checkpointer mechanism.

**a) `interrupt_before`** — static, declared at compile time. Simple, coarse-grained: always pause before a named node.


In [ ]:
gate_app = graph.compile(checkpointer=checkpointer, interrupt_before=["tools"])

result = gate_app.invoke({"messages": [("user", "Delete all records for user 42")]},
                          {"configurable": {"thread_id": "approval-1"}})
# Graph is now paused. Inspect the pending call before approving:
print(result["messages"][-1].tool_calls)

# Simulate a human approving the action, then resume by invoking with None:
gate_app.invoke(None, {"configurable": {"thread_id": "approval-1"}})

**b) `interrupt()`** — *(the modern, dynamic API — not shown in the source materials, which only cover `interrupt_before`)*. Called **inside a node**, so the pause condition can depend on runtime state (e.g. only pause for a specific tool, or only above a dollar threshold), not just "always pause before this node." Resume with `Command(resume=...)`, which lets the human's response flow back into the graph as data, not just a bare continue signal.


In [ ]:
from langgraph.types import interrupt, Command

def call_model_with_gate(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    if response.tool_calls and any(c["name"] == "calculator" for c in response.tool_calls):
        decision = interrupt({"pending_calls": response.tool_calls, "question": "Approve?"})
        if decision != "approve":
            return {"messages": [response.model_copy(update={"content": "Action cancelled by reviewer."})]}
    return {"messages": [response]}

gated_graph = StateGraph(MessagesState)
gated_graph.add_node("agent", call_model_with_gate)
gated_graph.add_node("tools", ToolNode(TOOLS))
gated_graph.add_edge(START, "agent")
gated_graph.add_conditional_edges("agent", tools_condition)
gated_graph.add_edge("tools", "agent")
gated_app = gated_graph.compile(checkpointer=checkpointer)

config2 = {"configurable": {"thread_id": "approval-2"}}
gated_app.invoke({"messages": [("user", "What's 12 * 7?")]}, config2)
# Graph paused inside the node at the interrupt() call. Resume, passing the human's decision
# back in as data:
gated_app.invoke(Command(resume="approve"), config2)

## 2.7 Streaming

Solves the manual delta-accumulation problem from Part 1's streaming section — LangGraph exposes streaming at multiple granularities, with tool calls already fully assembled per node.


In [ ]:
for event in app.stream({"messages": [("user", "What's the weather in Hanoi?")]}, stream_mode="updates"):
    print(event)   # one event per node completion — tool_calls already assembled, no delta accumulation

# stream_mode="values" yields the full state after each step (vs "updates", which yields only the delta).
async for event in app.astream_events({"messages": [("user", "What's the weather in Hanoi?")]}, version="v2"):
    if event["event"] == "on_tool_start":
        print("Calling tool:", event["name"])
    elif event["event"] == "on_chat_model_stream":
        pass  # token-level chunks — build a live UI from these without hand-accumulating deltas

## 2.8 Async End-to-End

*(Not covered in the source materials — every method above has an async twin: `.ainvoke()`, `.astream()`, `.astream_events()`, matching LangChain's async tool support from Part 1, section 1.10.)*


In [ ]:
async def run_graph_async():
    result = await app.ainvoke({"messages": [("user", "What's the weather in Tokyo?")]})
    return result["messages"][-1].content

print(await run_graph_async())  # in a .py script: asyncio.run(run_graph_async())

## 2.9 Multi-Tool, Multi-Case Patterns

- **Parallel tool execution** — `ToolNode` executes a turn's multiple tool calls concurrently by default.
- **Sequential dependency** — force ordering with `parallel_tool_calls=False` on the bound model, same as the raw-SDK notebook's section 1.5.
- **Retry with backoff around a flaky tool** — the decorator is framework-agnostic; the same wrapper works around a Part 1 raw function or a LangChain `@tool`.
- **Conditional tool availability** — bind a different subset of tools per node/state (e.g. an "admin" node with a `delete_record` tool a "support" node never sees) — least-privilege tool design implemented structurally, not just by prompt instruction.


In [ ]:
# Sequential dependency: force one tool call per turn
sequential_llm = llm.bind_tools(TOOLS, parallel_tool_calls=False)

# Conditional tool availability per node
admin_tools = TOOLS + [StructuredTool.from_function(
    func=lambda record_id: {"deleted": record_id},
    name="delete_record", description="Delete a record by ID. Admin-only.",
)]
admin_llm = llm.bind_tools(admin_tools)
support_llm = llm.bind_tools(TOOLS)  # no delete_record in scope — structurally can't call it

def admin_node(state: MessagesState):
    return {"messages": [admin_llm.invoke(state["messages"])]}

def support_node(state: MessagesState):
    return {"messages": [support_llm.invoke(state["messages"])]}
# Route to admin_node vs support_node based on an auth check in your graph's routing function.

## 2.10 Fan-Out with `Send` — Dynamic Parallel Branching

*(Not covered in the source materials — needed once "run this tool/subgraph once per item in a list" comes up, e.g. summarizing N documents in parallel and reducing the results.)* `Send` lets a routing function spawn a variable number of parallel node executions at runtime — a map step — whose outputs are then collected back into one state (the reduce step), all inside the same graph.


In [ ]:
from langgraph.types import Send

class FanOutState(TypedDict):
    cities: list[str]
    reports: Annotated[list, lambda a, b: a + b]  # reducer: collect results from every branch

def dispatch_weather_checks(state: FanOutState):
    # One Send per city -> LangGraph runs "check_one_city" once per city, in parallel
    return [Send("check_one_city", {"city": c}) for c in state["cities"]]

def check_one_city(state: dict):
    return {"reports": [get_weather(state["city"])]}

fanout_graph = StateGraph(FanOutState)
fanout_graph.add_node("check_one_city", check_one_city)
fanout_graph.add_conditional_edges(START, dispatch_weather_checks, ["check_one_city"])
fanout_graph.add_edge("check_one_city", END)
fanout_app = fanout_graph.compile()

result = fanout_app.invoke({"cities": ["hanoi", "tokyo", "london"], "reports": []})
print(result["reports"])  # three reports, computed concurrently, reduced back into one list

## 2.11 Multi-Agent Orchestration with Tools (Supervisor Pattern)

A LangGraph node doesn't have to be a single LLM call — it can be an entire compiled subgraph (e.g. another `create_react_agent`). This is how multi-agent systems are built: a supervisor routes to specialized sub-agents, each with its own scoped tool set, on top of the same tool-calling mechanics from Part 1.


In [ ]:
from langchain_core.messages import AIMessage

@tool
def search_web(query: str) -> str:
    """Simulated web search — replace with a real search API in production."""
    return f"Top result for '{query}': (simulated search snippet)"

research_agent = create_react_agent(llm, tools=[search_web])
writer_agent = create_react_agent(llm, tools=[])  # no tools — pure drafting from context

def researcher_node(state: MessagesState):
    result = research_agent.invoke(state["messages"])
    return {"messages": [result["messages"][-1]]}

def writer_node(state: MessagesState):
    result = writer_agent.invoke(state["messages"])
    return {"messages": [result["messages"][-1]]}

def route_to_next_agent(state: MessagesState):
    # Simple rule: if we haven't researched yet this turn, go research; otherwise write.
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and "research" not in str(last.content).lower():
        return "researcher"
    return "writer"

supervisor_graph = StateGraph(MessagesState)
supervisor_graph.add_node("researcher", researcher_node)
supervisor_graph.add_node("writer", writer_node)
supervisor_graph.add_edge(START, "researcher")
supervisor_graph.add_edge("researcher", "writer")
supervisor_graph.add_edge("writer", END)
supervisor_app = supervisor_graph.compile()

out = supervisor_app.invoke({"messages": [("user", "Research LangGraph and write a two-sentence summary.")]})
print(out["messages"][-1].content)

## 2.12 Choosing Your Layer — Comparison

| Concern | Raw SDK (Part 1 notebook) | LangChain `create_agent` | LangGraph |
| --- | --- | --- | --- |
| Understand every message on the wire | Full control | Partially hidden | Partially hidden |
| Provider portability (OpenAI/Anthropic/Gemini) | Manual per-provider code | Built in | Built in |
| Persistence across restarts | Build yourself | Not built in | Built in (`checkpointer`) |
| Pause for human approval | Build yourself | Not built in | Built in (`interrupt()` / `interrupt_before`) |
| Custom state beyond a message list | N/A | Not built in | Built in (`TypedDict` + reducers) |
| Dynamic parallel fan-out | Build yourself (`asyncio.gather`) | Not built in | Built in (`Send`) |
| Multi-agent graphs | Build yourself | Awkward | Native (subgraphs as nodes) |
| Observability | `print()` statements | LangSmith (opt-in) | LangSmith (opt-in) |
| Debuggability / minimal magic | Highest | Medium | Medium-high (explicit graph) |
| Recommended for | Learning, tiny scripts, ultra-custom control loops | Fast prototypes | Production agents |

## 2.13 Part 2 Recap

- LangChain's `@tool` / `bind_tools` are a thin, provider-agnostic layer over exactly the JSON Schema + `tool_calls` protocol from Part 1 — nothing new is invented.
- `create_agent` automates the `while` loop (it's a thin, opinionated layer over LangGraph itself) but you lose the ability to hand-design the graph shape unless you drop to LangGraph directly.
- LangGraph re-expresses the same loop as an explicit graph, and that explicitness is exactly what unlocks checkpointing, human-in-the-loop, custom state, dynamic fan-out, and multi-agent composition.
- Default to LangGraph for anything production-bound; reach for `create_agent` for the fastest possible prototype, or when its middleware system already covers what you need.

## Next Steps

Build the same calculator + weather-lookup agent you built with no framework, now compare three implementations side by side: (1) your raw-SDK `while` loop, (2) `create_agent` from section 1.8, (3) the hand-built LangGraph graph from section 2.1. Note exactly which lines of bookkeeping code disappeared at each step — that's the value the framework is buying you.
